# 리포트 53 — 운용 형상에서 경험 Pfa 를 재니 명목값의 1.52~2.66 배였다

> ### 한 일
> **운용 형상의 검출 사슬에서 거리-도플러 맵을 대량으로 다시 만들어 경험적 오경보율을 세고, 세 파형의 CFAR 문턱을 그 측정값에 맞춰 교정했다.**

### 결과
1. GPU 2717 s [^1] 동안 파형·모드마다 거리-도플러 맵 10,000 [^2]장을 돌려 경험 Pfa 를 측정했다.
2. 검출기 구현의 눈금을 먼저 확정했다 — 문턱 상수는 이론값과 상대오차 7.6e-16 [^3] 안에서 같고, 이상적 백색 맵 500,000 [^4]장(셀 564,000,000 [^5]개)에서 경험/명목 = 0.997 [^6] 다.
3. 운용 형상(CPI 프레임 48 [^7] · `g2x2_t6x6` · 0-도플러 마스크 1 [^8]빈)에서 명목 1e-04 [^9] 를 주면 WiFi 1.53 [^10]배 · LTE 2.66 [^11]배 · 5G 1.52 [^12]배로 울린다.
4. 그 형상의 교정표를 만들었다 — 경험 1e-04 [^9] 를 얻는 명목값은 WiFi 6.27e-05 [^13] · LTE 2.90e-05 [^14] · 5G 6.46e-05 [^15] 다.
5. 선행 census 16 [^16]편 · 전문 198 [^17]쪽에서 `CFAR` 와 `false alarm` 이 모두 0회인 논문이 13 [^18]편이고, 검출을 주장한 논문은 1 [^19]편이다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 경험 Pfa | 파형·명목값마다 거리-도플러 맵 10,000 [^2]장에 CA-CFAR 를 걸어 오경보 셀을 세었다 (GPU, 2717 s [^1]) |
| 문턱 상수 | CA-CFAR α 를 이론식과 대조했다 — 상대오차 7.6e-16 [^3] · 훈련셀 264 [^20]개 |
| 측정 구간 | 명목 1e-06 [^21] ~ 1e-02 [^22] 아홉 점. 그 밖의 운용점은 외삽이라 표에서 뺀다 |
| 교정표 | 측정한 명목–경험 곡선을 역보간한다. 자유공간 기하는 형상이 달라 `src/freespace_detect.py:711` 이 거기서 다시 잰다 |
| 왜 통제 시뮬레이션인가 | 오경보율을 명목값과 대조하려면 같은 배경을 수만 번 다시 만들어 세어야 한다. 실외 실측은 배경을 주어진 대로 받는다 |

### 재현

```bash
cd /workspace/sionna
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_cfar.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_cfar.json`, `outputs/prior_census.json` |
| 소요 | CFAR 측정이 2717 s [^1] (GPU 1장) |
| 비고 | 맵 수는 `--maps` / `--white` 로 줄인다. 줄이면 신뢰구간이 넓어진다. |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 51 «수신 → ECA → 거리도플러 → CFAR»](51_chain.ipynb) | 사슬의 네 단계와 파형별 형상 |

---

## 왜 배경을 다시 만드는가

⭐ 오경보율을 명목값과 대조하려면 같은 배경을 수만 번 다시 만들어 세어야 한다. 통제 시뮬레이션이 그 일을 한다.

선행 census 16 [^16]편 · 전문 198 [^17]쪽에서 `CFAR` 와 `false alarm` 이 모두 0회인 논문이 13 [^18]편이고, 검출을 주장한 논문은 1 [^19]편이다. OpenISAC(`arXiv:2601.03535v2`, preprint)은 전문 16 [^23]쪽에서 `CFAR` 0 [^24]회 · `false alarm` 0 [^25]회 · `detection probability` 0 [^26]회다.

## 눈금부터 확정한다

이 절의 모든 수는 **운용 형상** 하나에서 나온다 — DPI+ECA · 운용 거리창 · CPI 프레임 48 [^7] · 훈련창 `g2x2_t6x6` · 0-도플러 마스크 1 [^8]빈.

검출기 구현이 먼저 맞아야 배율을 사슬 탓으로 돌릴 수 있다. 문턱 상수는 이론값과 상대오차 7.6e-16 [^3] 안에서 같고, 잡음 추정/실제 전력 = 1.000 [^27] 다. 이상적 백색 맵 500,000 [^4]장에서 경험/명목 = 0.997 [^6] 로 눈금이 1 에 선다.

![f4_pfa](../outputs/figures/report04_f4_pfa.png)

**그림 1.** 명목 Pfa 를 요구하면 실제로는 몇 배가 울리는가?

## 운용 형상 교정표

운용 명목값은 1e-04 [^9] 다. 왼쪽 열이 그 값에서 측정된 배율이고, 오른쪽 열이 교정된 명목값이다.

| 파형 | 명목 1e-4 에서 경험/명목 | 경험 1e-4 를 얻을 명목 Pfa |
|---|---|---|
| WiFi | 1.53 [^10]배 | 6.27e-05 [^13] |
| LTE | 2.66 [^11]배 | 2.90e-05 [^14] |
| 5G | 1.52 [^12]배 | 6.46e-05 [^15] |

세 파형의 배율이 서로 다르다. 교정이 셋을 같은 실제 오경보율 위에 올린다.

`src/passive_process.py:283` 이 이 JSON 을 읽고, `pfa_nominal_for()`(`src/passive_process.py:338`)가 파형별 명목값을 돌려준다.

## 이 표를 읽는 곳

`src/experiment_detection.py:358` 과 `src/experiment_x410.py:175` 가 `src/passive_process.py:283` 을 거쳐 이 표를 읽는다.

교정이 없으면 세 파형 비교가 서로 다른 실제 오경보율 위에서 이뤄진다. 그 위에서 선 비교가 [편 58 «자유공간 형상에서 문턱을 다시 재니 세 밴드가 SNR90 하나를 공유한다»](58_shared-threshold.ipynb) 다.

이 표는 **레퍼런스 채널이 이상적일 때**의 형상에서 잰 값이다. 레퍼런스가 오염되면 거리-도플러 맵의 통계가 달라져 이 교정이 그대로 서지 않는다 — 그 형상에서 팔마다 경험 Pfa 를 다시 센 것이 [리포트 8-2 «기준채널이 현실이면 얼마를 잃는가»](08_2_two_channel.ipynb) 다.

배율이 왜 1 이 아닌지, 이 표가 어디까지 쓰이는지는 [편 54 «그 배율의 원인은 셀 상관이고, 교정표는 형상마다 다시 재야 한다»](54_cfar-why.ipynb) 가 든다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 실외 클러터 배경 위에서 같은 Pfa 스윕을 돌린다 | 교정 배율이 배경에 따라 얼마나 움직이는지 수치로 확정된다 | `benchmark/verify_cfar.py` → [편 67 «X410 의 12-bit ADC 동적범위가 직…»](67_hardware.ipynb) |
| 맵 수를 한 자릿수 올려 명목 1e-06 [^28] 구간까지 측정한다 | 저 Pfa 운용점의 교정값이 측정 구간 안으로 들어온다 | `benchmark/verify_cfar.py --maps` · `calib_op_mask1.points` |
| 표적 σ 를 앵커 위에서 읽어 Pd 절대값을 다시 푼다 | Pd 절대값이 교정된 Pfa 와 같은 근거 위에 선다 | [편 60 «앵커 σ 위의 R90 은 비교가능 12칸에서…»](60_r90.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 28개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/verify_cfar.json` | `meta.runtime_s` | 2717 |
| [^2] | `outputs/verify_cfar.json` | `meta.n_maps_chain` | 10000 |
| [^3] | `outputs/verify_cfar.json` | `alpha_audit.g2x2_t6x6.rel_err` | 7.581e-16 |
| [^4] | `outputs/verify_cfar.json` | `meta.n_maps_white` | 500000 |
| [^5] | `outputs/verify_cfar.json` | `white.48x24.rows[89].cells` | 564000000 |
| [^6] | `outputs/verify_cfar.json` | `white.48x24.rows[89].ratio` | 0.9966 |
| [^7] | `outputs/verify_cfar.json` | `meta.M_cpi` | 48 |
| [^8] | `outputs/verify_cfar.json` | `meta.zd_mask_operational` | 1 |
| [^9] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.op.rows[89].pfa_nom` | 0.0001 |
| [^10] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.op.rows[89].ratio` | 1.531 |
| [^11] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.op.rows[89].ratio` | 2.663 |
| [^12] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.op.rows[89].ratio` | 1.521 |
| [^13] | `outputs/verify_cfar.json` | `chain.WiFi80.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed` | 6.27e-05 |
| [^14] | `outputs/verify_cfar.json` | `chain.LTE20.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed` | 2.905e-05 |
| [^15] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed` | 6.46e-05 |
| [^16] | `outputs/prior_census.json` | `meta.n_papers` | 16 |
| [^17] | `outputs/prior_census.json` | `counts.total_pages` | 198 |
| [^18] | `outputs/prior_census.json` | `counts.zero_cfar_and_falsealarm` | 13 |
| [^19] | `outputs/prior_census.json` | `counts.claims_detection` | 1 |
| [^20] | `outputs/verify_cfar.json` | `alpha_audit.g2x2_t6x6.N_interior` | 264 |
| [^21] | `outputs/verify_cfar.json` | `meta.pfa_nominal[8]` | 1e-06 |
| [^22] | `outputs/verify_cfar.json` | `meta.pfa_nominal[0]` | 0.01 |
| [^23] | `outputs/prior_census.json` | `papers[14].pages` | 16 |
| [^24] | `outputs/prior_census.json` | `papers[14].terms.cfar` | 0 |
| [^25] | `outputs/prior_census.json` | `papers[14].terms.false_alarm` | 0 |
| [^26] | `outputs/prior_census.json` | `papers[14].terms.detection_probability` | 0 |
| [^27] | `outputs/verify_cfar.json` | `alpha_audit.g2x2_t6x6.noise_est_over_power` | 1 |
| [^28] | `outputs/verify_cfar.json` | `chain.NR100.dpi_eca.calib_op_mask1.points[4].pfa_target_emp` | 1e-06 |